# Plant Disease Detection - Prediction Notebook

Bu notebook, eğitilmiş **PlantVillage** ve **PlantDoc** modellerini kullanarak yeni görüntüler üzerinde tahmin yapmak için kullanılır.

## Özellikler

- 📸 Bir resim yolunu ver
- 🔁 32×32 griye çevir
- 🧠 `model_pv` (PlantVillage) ve `model_pd` (PlantDoc) ile tahmin al
- 📝 (PlantVillage için) sınıf adını yazdır


## 1. PlantVillage Sınıf İsimlerini Geri Alalım

Bu adım, tahmini **insan okunur** hale getirmek için gereklidir (klasör isimleriyle).

> Bu fonksiyon, PlantVillage preprocessing'te kullandığımız mantıkla **aynı**: klasör isimlerini okuyup alfabetik sıralıyor.


In [2]:
import os

def get_class_names(root_dir):
    """
    Return sorted list of class folder names under root_dir.
    """
    names = []
    for item in os.listdir(root_dir):
        full_path = os.path.join(root_dir, item)
        if os.path.isdir(full_path):
            names.append(item)
    names = sorted(names)
    return names

# PlantVillage root (Source Code klasöründen bakınca)
PLANTVILLAGE_ROOT = "../Dataset/plantvillage/color"  # Klasör yapınıza göre "../Datasets/plantvillage/color" olabilir

class_names_pv = get_class_names(PLANTVILLAGE_ROOT)
idx_to_class_pv = {idx: name for idx, name in enumerate(class_names_pv)}

print("Number of PlantVillage classes:", len(class_names_pv))
print("First 5 class names:", class_names_pv[:5])


Number of PlantVillage classes: 38
First 5 class names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


> **Önemli not:** Preprocessing'te de sınıfları alfabetik sıralayıp indexlemişsek (öyle yapmıştık), buradaki mapping **eğitimde kullanılan y ile uyumlu** olur.


## 2. `load_image_as_vector` Fonksiyonu

Eğer bu fonksiyon zaten yukarıda tanımlıysa, tekrar yazmana gerek yok. Yoksa şunu ekle:


In [3]:
from PIL import Image

IMG_SIZE = 32  # aynı boyut

def load_image_as_vector(path, img_size=IMG_SIZE):
    """
    Load an image file, convert it to grayscale, resize to img_size x img_size,
    normalize pixel values to [0, 1], and flatten to a 1D list (feature vector).
    """
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())  # length = img_size * img_size
        vector = [p / 255.0 for p in pixels]
        return vector


## 3. Test Fonksiyonları: Tek Bir Fotoğrafı 2 Modele Soralım

Burada:
- Resim yolunu veriyorsun (`image_path`)
- Aynı vektörü hem `model_pv` hem `model_pd` üzerinde kullanıyoruz.

> **Not:** Burada `model_pv` ve `model_pd` zaten notebook'ta eğittiğin modeller olmalı (yani daha önce `model_pv.fit(...)`, `model_pd.fit(...)` çalışmış olmalı).


In [4]:
def predict_with_plantvillage(image_path):
    """
    Predict class for a given image using PlantVillage model (model_pv).
    Returns predicted index and class name.
    """
    x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
    pred_idx = model_pv.predict_one(x_vec)
    class_name = idx_to_class_pv.get(pred_idx, f"unknown_{pred_idx}")
    print(f"[PlantVillage] predicted class index = {pred_idx}")
    print(f"[PlantVillage] predicted class name  = {class_name}")
    return pred_idx, class_name


def predict_with_plantdoc(image_path):
    """
    Predict class for a given image using PlantDoc model (model_pd).
    Returns predicted index (internal PlantDoc index).
    """
    x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
    pred_idx = model_pd.predict_one(x_vec)
    print(f"[PlantDoc] predicted class index (internal) = {pred_idx}")
    # Şimdilik sadece index yazıyoruz. İstersen ileride bu index'i gerçek
    # PlantDoc sınıf ismine map eden bir tablo ekleyebiliriz.
    return pred_idx


## 4. Fotoğrafla Test Etme

Şimdi bir yaprak fotoğrafını proje içine bir yere koy (örneğin):

```text
ProjectFolder/
   TestImages/
       my_tomato_leaf.jpg
```

Notebook'tan bakınca yolun mesela şöyle olabilir:

* Notebook `Source Code` klasöründeyse:
  `../TestImages/my_tomato_leaf.jpg`


In [5]:
test_image_path = "../TestImages/apple_healty.jpg"

print("Testing with PlantVillage model:")
pv_idx, pv_name = predict_with_plantvillage(test_image_path)

print("\nTesting with PlantDoc model:")
pd_idx = predict_with_plantdoc(test_image_path)


Testing with PlantVillage model:


NameError: name 'model_pv' is not defined

### Örnek Çıktı

Çıktı örneği gibi bir şey göreceksin:

```text
[PlantVillage] predicted class index = 12
[PlantVillage] predicted class name  = Tomato___Late_blight

[PlantDoc] predicted class index (internal) = 5
```

Şu an için PlantDoc tarafında sadece index gösteriyoruz; istersen bir sonraki adımda PlantDoc class id → class name mapping'ini de ekleyip oradan "Bacterial Spot", "Leaf Mold" vs. şeklinde yazdırabiliriz.


## 5. Özet

* **Evet**, aynı fotoğrafı hem PlantVillage modeline hem PlantDoc modeline sorabilirsin.
* Bunun için tek gereken:
  * Aynı preprocessing (`load_image_as_vector`)
  * `model_pv.predict_one(...)` ve `model_pd.predict_one(...)`
  * PlantVillage için `idx_to_class_pv` ile sınıf adını geri almak

### Sonraki Adımlar

İstersen bir sonraki adımda:
* Bir örnek çıktı (tahmin edilen sınıf isimleri) alın
* Bunu raporunuzda "Case Study / Qualitative Examples" gibi bir alt başlıkta nasıl anlatacağını yazabilirsiniz
